# 🔍 PyTorch CNN Deep Dive: Conv2d, MaxPool2d ve Boyut Hesaplama Rehberi

Bu notebook, **Evrişimli Sinir Ağları'nın (Convolutional Neural Networks - CNN)** temel yapı taşlarını, 4 boyutlu tensör mekaniğini ve katmanlar arasındaki **şekil dönüşümlerini (shape transformations)** deneysel olarak adım adım inceler.

### 🎯 Bu Rehberde Yanıtlanan Temel Sorular:
1. Bir görsel bilgisayarda neden 4 boyutlu `[Batch, Channels, Height, Width]` olarak temsil edilir?
2. `nn.Conv2d` katmanındaki `in_channels`, `out_channels`, `kernel_size`, `stride` ve `padding` parametreleri gerçekte ne yapar?
3. Katman çıktısının piksel boyutu matematiksel olarak nasıl hesaplanır? ($O = \lfloor\frac{W - K + 2P}{S}\rfloor + 1$)
4. `nn.MaxPool2d` neden kullanılır ve uzamsal boyutları nasıl yarıya indirir?
5. **TinyVGG** mimarisinde $32 \times 32$'lik bir görsel nasıl $8 \times 8$'e iner ve en sondaki `nn.Linear`'a neden `hidden_units * 8 * 8` girdisi verilir?

---
*Görselleştirme için önerilen interaktif araç:* [CNN Explainer (Polo Club)](https://poloclub.github.io/cnn-explainer/)


## 1. Kütüphanelerin Yüklenmesi


In [1]:
import torch
from torch import nn
import torchvision
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt

## 2. 4 Boyutlu Görsel Tensörü Anatomisi `[B, C, H, W]`

Geleneksel tablosal derin öğrenmede girdilerimiz `[Batch_Size, Feature_Count]` şeklindeydi. Ancak bilgisayarlı görüde (Computer Vision) bir görsel 4 boyutlu bir tensör olarak temsil edilir:
- **`B` (Batch Size):** Aynı anda işlenen görsel adedi (örn. 32).
- **`C` (Channels):** Renk kanalları (Renkli RGB için **3**, Siyah-Beyaz Grayscale için **1**).
- **`H` (Height):** Görselin piksel cinsinden yüksekliği (örn. 32).
- **`W` (Width):** Görselin piksel cinsinden genişliği (örn. 32).


In [2]:
torch.manual_seed(42)

# CIFAR-10 verimizi temsil eden örnek bir tensör (32'lik mini-batch)
images = torch.randn(size=(32, 3, 32, 32)) # [Batch_Boyutu, Renk_Kanalı, Yükseklik, Genişlik]
test_image = images[0] 
print(f"Image batch shape: {images.shape} -> [Batch_Boyutu, Renk_Kanalı, Yükseklik, Genişlik]")
print(f"Tekil görsel şekli: {test_image.shape} -> [Renk_Kanalı, Yükseklik, Genişlik]") 
print(f"Tekil görsel piksel değerleri:\n{test_image}")

Image batch shape: torch.Size([32, 3, 32, 32]) -> [batch_size, color_channels, height, width]
Single image shape: torch.Size([3, 32, 32]) -> [color_channels, height, width]
Single image pixel values:
tensor([[[ 1.9269e+00,  1.4873e+00,  9.0072e-01,  ...,  4.1759e-02,
          -2.5158e-01,  8.5986e-01],
         [-1.3847e+00, -8.7124e-01, -2.2337e-01,  ...,  1.8446e+00,
          -1.1845e+00,  1.3835e+00],
         [ 1.4451e+00,  8.5641e-01,  2.2181e+00,  ..., -8.2777e-01,
           1.3347e+00,  4.8354e-01],
         ...,
         [ 5.1823e-02, -3.2848e-01, -2.2472e+00,  ...,  1.4557e+00,
          -3.4610e-01, -2.6338e-01],
         [-4.4770e-01, -7.2882e-01, -1.6066e-01,  ...,  5.4047e-01,
           4.3507e-01, -2.2717e+00],
         [-1.3386e-01, -5.8557e-02,  1.2574e-01,  ...,  1.1085e+00,
           5.5442e-01,  1.5818e+00]],

        [[-1.2248e+00,  9.6289e-01, -1.5785e+00,  ...,  7.8247e-01,
          -6.4659e-02, -2.2984e-04],
         [ 6.8309e-01,  1.0637e-01,  3.5032e-01, 

## 3. `nn.Conv2d` Katman Parametrelerinin Anatomisi

`nn.Conv2d` katmanının en kritik 5 parametresi şunlardır:

| Parametre | Anlamı | Açıklama |
| :--- | :--- | :--- |
| **`in_channels`** | Giriş Kanal Sayısı | Giren görselin renk katmanı adedi (RGB için 3, Siyah-Beyaz için 1). |
| **`out_channels`** | Çıkış Kanal Sayısı | Resim üzerinde gezecek **bağımsız özellik dedektörü (filtre) sayısı**. |
| **`kernel_size`** | Filtre Boyutu | Resim üzerinde kaydırılan matrisin boyutu (Genellikle $3 \times 3$ veya $5 \times 5$). |
| **`stride`** | Adım Boyu | Filtrenin pikseller üzerinde kaçar piksel atlayarak kayacağı (Varsayılan: 1). |
| **`padding`** | Kenar Dolgusu | Kenar piksellerin kaybolmasını önlemek için resmin etrafına eklenen sıfır pikselleri. |

> **⚠️ Sık Yapılan Hata (RuntimeError):**  
> `in_channels` değerine görselin düzleştirilmiş toplam piksel sayısı ($3 \times 32 \times 32 = 3072$) yazılırsa PyTorch hata verir. Çünkü `Conv2d` düzleştirilmiş vektör değil, **kanal sayısını (3)** bekler!


In [4]:
#torch.manual_seed(42)

#conv_layer = nn.Conv2d(in_channels=3072,
 #                      out_channels=10,
 #                      kernel_size=3,
 #                      stride=1,
 #                      padding=0) 

#conv_layer(test_image)
#bu 3072 verildiğinde hata verir
# PyTorch Hata Mesajı:
# RuntimeError: Given groups=1, weight of size [10, 3072, 3, 3], expected input[1, 3, 32, 32] to have 3072 channels, but got 3 channels instead
# Açıklama: Model 3072 kanal bekledi ama görselde sadece 3 renk kanalı var!


### Deney 1: `kernel_size=3`, `stride=1`, `padding=0`
Görseli dolgu olmadan (`padding=0`) $3 \times 3$'lük filtreden geçiriyoruz. Filtre kenarlardan taştığı için görsel $32 \times 32$'den **$30 \times 30$**'a sıkışır.


In [5]:
torch.manual_seed(42)
# Bir evrişim (Conv2d) katmanı oluşturup görseli içinden geçirdiğimizde boyutların nasıl değiştiğini inceleyelim
# (Parametreleri değiştirerek çıktı boyutlarını test edebilirsiniz)
conv_layer = nn.Conv2d(in_channels=3,
                       out_channels=10,
                       kernel_size=3,
                       stride=1,
                       padding=0) #padding=0: Kenar dolgusu olmadan görsel taranır 

# Görseli evrişim katmanından geçiriyoruz
conv_layer(test_image)

tensor([[[ 1.1667e+00,  1.4325e-01,  1.6771e-02,  ...,  3.1925e-02,
          -7.1931e-02,  3.7874e-01],
         [ 9.5715e-01,  3.2787e-01,  2.6548e-01,  ..., -3.0249e-01,
          -1.8824e-01, -3.8694e-01],
         [ 2.7237e-02,  4.1275e-01,  1.0648e+00,  ..., -5.8316e-01,
          -7.7496e-01,  7.0928e-01],
         ...,
         [-7.4959e-01, -5.7313e-01,  2.3699e-01,  ...,  8.5081e-01,
           2.3852e-01, -4.1881e-01],
         [-5.5988e-01,  9.8861e-02, -8.2409e-01,  ..., -4.4532e-02,
           2.9962e-01, -2.9472e-02],
         [ 1.2873e-01, -7.5458e-01, -4.0582e-01,  ..., -4.7212e-02,
           1.3111e+00,  6.3414e-01]],

        [[ 3.3298e-01,  3.3936e-01, -6.4400e-01,  ...,  1.4035e-01,
          -6.6302e-01, -5.8043e-03],
         [ 5.9086e-01, -1.1782e+00,  5.4756e-01,  ..., -5.9479e-01,
          -5.3999e-01, -4.8962e-01],
         [-1.6372e+00,  1.5448e-01,  2.5152e-01,  ..., -6.4175e-01,
          -1.3647e-01, -3.4338e-01],
         ...,
         [-6.1613e-01, -1

In [6]:
conv_layer(test_image).shape

torch.Size([10, 30, 30])

In [7]:
test_image.shape

torch.Size([3, 32, 32])

## 4. Evrişim Çıktısı Boyut Formülü (Spatial Dimension Formula)

Bir evrişim katmanının (`Conv2d`) ardından en veya boyun kaça düşeceği şu formülle hesaplanır:

$$\text{Output} = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$

* **$W$ (Input Dimension):** Orijinal genişlik veya yükseklik (örn. 32)
* **$K$ (Kernel Size):** Filtre boyutu (örn. 3 veya 5)
* **$P$ (Padding):** Kenar dolgusu (0, 1, 2...)
* **$S$ (Stride):** Adım boyutu (1, 2...)

### Deney 2: `kernel_size=5`, `stride=2`, `padding=0`
Formülü test edelim:
$$\text{Output} = \left\lfloor \frac{32 - 5 + 2(0)}{2} \right\rfloor + 1 = \left\lfloor \frac{27}{2} \right\rfloor + 1 = 13 + 1 = 14$$
Piksel boyutunun tam olarak **14**'e düştüğünü görelim:


In [17]:
# Filtre parametrelerini değiştirdiğimizde piksel boyutunun nasıl değiştiğini görelim
torch.manual_seed(42)
# Farklı parametrelere sahip yeni bir Conv2d katmanı oluşturalım 
conv_layer_2 = nn.Conv2d(in_channels=3, # Giriş görselimizle aynı sayıda renk kanalı (RGB için 3)
                         out_channels=10,
                         kernel_size=(5, 5), # Kernel genellikle kare matristir, (5, 5) şeklinde tuple da verilebilir
                         stride=2,
                         padding=0)

# Tekil görseli yeni conv_layer_2 katmanından geçiriyoruz 
conv_layer_2(test_image).shape

torch.Size([10, 14, 14])

## 5. `nn.MaxPool2d` ile Boyutsal İndirgeme (Downsampling)

`nn.MaxPool2d(kernel_size=2)` katmanı:
- $2 \times 2$'lik penceredeki **en yüksek (en baskın) aktivasyon değerini** seçer.
- Kanal sayısına (`channels`) **kesinlikle dokunmaz**.
- Uzamsal boyutları (en ve boy) **tam yarıya indirir** ($30 \times 30 \to 15 \times 15$ veya $32 \times 32 \to 16 \times 16$).
- Modelin hesaplama yükünü azaltır ve küçük kaymalara karşı modeli dayanıklı hale getirir (Translation Invariance).


In [20]:
# Görselin orijinal ve batch boyutu eklenmiş (unsqueeze) şekillerini yazdıralım
print(f"Test image original shape: {test_image.shape}")
print(f"Test image with unsqueezed dimension: {test_image.unsqueeze(dim=0).shape}")

# Örnek bir nn.MaxPool2d katmanı oluşturalım
max_pool_layer = nn.MaxPool2d(kernel_size=2)

# Veriyi önce evrişim katmanından geçiriyoruz
test_image_through_conv = conv_layer(test_image.unsqueeze(dim=0))
print(f"Conv2d katmanından geçtikten sonraki şekil: {test_image_through_conv.shape}")

# Ardından havuzlama (MaxPool) katmanından geçiriyoruz
test_image_through_conv_and_max_pool = max_pool_layer(test_image_through_conv)
print(f"Conv2d ve MaxPool2d sonrası şekil: {test_image_through_conv_and_max_pool.shape}")

Test image original shape: torch.Size([3, 32, 32])
Test image with unsqueezed dimension: torch.Size([1, 3, 32, 32])
Shape after going through conv_layer(): torch.Size([1, 10, 30, 30])
Shape after going through conv_layer() and max_pool_layer(): torch.Size([1, 10, 15, 15])


## 6. TinyVGG Blokları Boyunca Adım Adım Tensör Dönüşüm Haritası

TinyVGG mimarisi iki ardışık konvolüsyon bloğundan ve bir sınıflandırıcıdan oluşur:
- **Block 1:** `Conv2d (pad=1)` $\to$ `ReLU` $\to$ `Conv2d (pad=1)` $\to$ `ReLU` $\to$ `MaxPool2d(2)`
- **Block 2:** `Conv2d (pad=1)` $\to$ `ReLU` $\to$ `Conv2d (pad=1)` $\to$ `ReLU` $\to$ `MaxPool2d(2)`
- **Classifier:** `Flatten()` $\to$ `Linear()`

`padding=1` ve `kernel_size=3` seçildiği için Conv katmanları boyutu $32 \times 32$ olarak korur. Boyut indirgemesini sadece `MaxPool` yapar!

Tensörün adım adım yolculuğunu canlı izleyelim:


In [14]:
x = torch.randn(1,3,32,32)
print("Input: ", x.shape)

conv1 = nn.Conv2d(in_channels=3, 
                      out_channels=10,
                      kernel_size=3,
                      stride=1,
                      padding=1)

conv2 = nn.Conv2d(in_channels=10, 
                      out_channels=10,
                      kernel_size=3,
                      stride=1,
                      padding=1)

pool1 = nn.MaxPool2d(kernel_size = 2, stride =2)

x = conv1(x)
print("after conv1:",x.shape)

x = conv2(x)
print("after conv2:",x.shape)

x = pool1(x)
print("after pool1:",x.shape)


conv3 = nn.Conv2d(in_channels=10, 
                      out_channels=10,
                      kernel_size=3,
                      stride=1,
                      padding=1)

conv4 = nn.Conv2d(in_channels=10, 
                      out_channels=10,
                      kernel_size=3,
                      stride=1,
                      padding=1)

pool2 = nn.MaxPool2d(kernel_size = 2, stride =2)

x = conv3(x)
print("after conv3:",x.shape)

x = conv4(x)
print("after conv4:",x.shape)

x = pool2(x)
print("after pool2:",x.shape)

Input:  torch.Size([1, 3, 32, 32])
after conv1: torch.Size([1, 10, 32, 32])
after conv2: torch.Size([1, 10, 32, 32])
after pool1: torch.Size([1, 10, 16, 16])
after conv3: torch.Size([1, 10, 16, 16])
after conv4: torch.Size([1, 10, 16, 16])
after pool2: torch.Size([1, 10, 8, 8])


## 7. `nn.Flatten` ve Sınıflandırıcıya (`nn.Linear`) Bağlantı: `hidden_units * 8 * 8` Gizemi

Yukarıdaki çıktıya dikkat edelim:
- 1. Havuz sonrası boyut: `[1, 10, 16, 16]`
- 2. Havuz sonrası boyut: `[1, 10, 8, 8]`

Son katmanda elimizde her biri $8 \times 8$ piksel olan $10$ adet özellik haritası (kanal) var.  
Sınıflandırma katmanı (`nn.Linear`) 3 boyutlu tensörleri kabul etmediği için **`nn.Flatten()`** bu küpü tek bir vektöre açar:

$$\text{in\_features} = \text{hidden\_units} \times 8 \times 8 = 10 \times 8 \times 8 = 640$$

Bu yüzden `CIFAR10ClassifierCNN` modelindeki sınıflandırıcı katmanı şu şekilde tanımlanır:

```python
self.classifier = nn.Sequential(
    nn.Flatten(),
    nn.Linear(in_features=hidden_units * 8 * 8, out_features=output_shape)
)
```


In [15]:
10 * 8 * 8

640

### 📌 Özet: CNN Boyut Takip Kuralları
1. **`padding=1`, `kernel_size=3`, `stride=1`** kombinasyonu uzamsal boyutu ($H, W$) asla değiştirmez.
2. **`nn.MaxPool2d(kernel_size=2)`** uzamsal boyutu ($H, W$) tam ikiye böler ($32 \to 16 \to 8$).
3. **`out_channels`** bir sonraki katmanın **`in_channels`** değeri olmak zorundadır.
4. **`Flatten()`** sonrası giriş sayısı daima $\text{Kanal Sayısı} \times H_{son} \times W_{son}$ formülüyle bulunur.
